<a href="https://colab.research.google.com/github/santidrelew/tp-mpg/blob/main/notebooks/03_OLS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3 - Regresión con OLS

In [50]:
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    !git clone https://github.com/santidrelew/tp-mpg.git
    %cd tp-mpg

In [51]:
# Importamos todas las librerías necesarias
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.formula.api as smf


In [52]:
# Cargamos los datasets
PROJECT_DIR = Path().resolve() if EN_COLAB else Path().resolve().parent
DATA_BASE_DIR = PROJECT_DIR / "datos" / "procesados"

df = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_filtrado.csv")
df_muestra = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_muestra.csv")

In [53]:
TARGET = "log_daily_price"
FEATURES = [c for c in df.columns if c != TARGET]
CATEGORICAS = ["neighbourhood_cleansed"]

Creamos una unica funcion para usar en los 4 casos y no tener que repetir codigo:

In [54]:
def construir_formula(target: str, variables: list, categoricas: list) -> str:
    terminos = [f"C({v})" if v in categoricas else v for v in variables]
    return f"{target} ~ " + " + ".join(terminos)


def calcular_metricas(modelo, df_data: pd.DataFrame, target: str) -> dict:
    y_true = df_data[target]
    y_pred = modelo.predict(df_data)

    return {
        "N Variables": int(modelo.df_model),
        "R2": r2_score(y_true, y_pred),
        "R2 Ajustado": modelo.rsquared_adj,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    }

Primero, hacemos la regresion con todas las variables EDA de nuestro dataset entero:

In [55]:
formula_completa = construir_formula(TARGET, FEATURES, CATEGORICAS)
modelo_completo = smf.ols(formula=formula_completa, data=df).fit()

print(modelo_completo.summary())

                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.395
Model:                            OLS   Adj. R-squared:                  0.393
Method:                 Least Squares   F-statistic:                     183.4
Date:                Wed, 11 Nov 2026   Prob (F-statistic):               0.00
Time:                        00:30:30   Log-Likelihood:                -5416.8
No. Observations:                9297   AIC:                         1.090e+04
Df Residuals:                    9263   BIC:                         1.114e+04
Df Model:                          33                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

En este analisis, podemos ver que las siguientes 6 variables no aportan evidencia estadística suficiente para rechazar la hipotesis nula y sus intervalos de confianza cruzan el cero:
* minimum_nights (p = 0.171)
* Washer (p = 0.139)
* Paid_parking_on_premises (p = 0.960)
* Long_term_stays_allowed (p = 0.230)
* Smoking_allowed (p = 0.219)
* check_in_24_hour (p = 0.252)

Ahora, haremos una nueva regresion sobre el dataset EDA completo pero omitiendo estas variables:

In [56]:
variables_no_significativas = ["minimum_nights", "Washer", "Paid_parking_on_premises", "Long_term_stays_allowed", "Smoking_allowed", "check_in_24_hour"]
variables_significativas = [cols for cols in FEATURES if cols not in variables_no_significativas]
formula_reducida_significativa = construir_formula(TARGET, variables_significativas, CATEGORICAS)
modelo_reducido_significativo = smf.ols(formula=formula_reducida_significativa, data=df).fit()

print(modelo_reducido_significativo.summary())

                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.395
Model:                            OLS   Adj. R-squared:                  0.393
Method:                 Least Squares   F-statistic:                     223.8
Date:                Wed, 11 Nov 2026   Prob (F-statistic):               0.00
Time:                        00:30:30   Log-Likelihood:                -5421.1
No. Observations:                9297   AIC:                         1.090e+04
Df Residuals:                    9269   BIC:                         1.110e+04
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

Ahora vamos a hacer el mismo procedimiento con la muestra seleccionada:

In [57]:
modelo_completo_muestra = smf.ols(formula=formula_completa, data=df_muestra).fit()

print(modelo_completo_muestra.summary())

                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.431
Model:                            OLS   Adj. R-squared:                  0.405
Method:                 Least Squares   F-statistic:                     16.42
Date:                Wed, 11 Nov 2026   Prob (F-statistic):           5.74e-67
Time:                        00:30:30   Log-Likelihood:                -373.95
No. Observations:                 750   AIC:                             815.9
Df Residuals:                     716   BIC:                             973.0
Df Model:                          33                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

Vemos que las variables no significativas son muchas mas que en el modelo entrenado sobre el dataset completo:

* (ESCRIBIR CADA VARIABLE)

Ahora, haremos una nueva regresion sobre la muestra pero omitiendo estas variables:

In [58]:
variables_significativas_muestra = ["neighbourhood_cleansed", "host_total_listings_count", "accommodates", "bathrooms", "bedrooms", "number_of_reviews", "review_scores_rating", "WiFi"]
formula_reducida_significativa_muestra = construir_formula(TARGET, variables_significativas_muestra, CATEGORICAS)
modelo_reducido_significativo_muestra = smf.ols(formula=formula_reducida_significativa_muestra, data=df_muestra).fit()

print(modelo_reducido_significativo_muestra.summary())

                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.412
Model:                            OLS   Adj. R-squared:                  0.400
Method:                 Least Squares   F-statistic:                     34.34
Date:                Wed, 11 Nov 2026   Prob (F-statistic):           1.55e-74
Time:                        00:30:30   Log-Likelihood:                -385.89
No. Observations:                 750   AIC:                             803.8
Df Residuals:                     734   BIC:                             877.7
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

Ahora comparemos las 4 regresiones que hicimos:

In [59]:
res_completo_pob = calcular_metricas(modelo_completo, df, TARGET)
res_reducido_pob = calcular_metricas(modelo_reducido_significativo, df, TARGET)
res_completo_mue = calcular_metricas(modelo_completo_muestra, df_muestra, TARGET)
res_reducido_mue = calcular_metricas(modelo_reducido_significativo_muestra, df_muestra, TARGET)

tabla_comparativa = pd.DataFrame(
    [
        res_completo_pob,
        res_reducido_pob,
        res_completo_mue,
        res_reducido_mue,
    ],
    index=[
        "1. Poblacion (Modelo completo)",
        "2. Población (Modelo significativo)",
        "3. Muestra (Modelo completo)",
        "4. Muestra (Modelo significativo)",
    ],
)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
tabla_comparativa

,N Variables,R2,R2 Ajustado,MAE,RMSE
1. Poblacion (Modelo completo),33,0.3952,0.3930,0.3107,0.4333
2. Población (Modelo significativo),27,0.3946,0.3928,0.3109,0.4335
3. Muestra (Modelo completo),33,0.4308,0.4046,0.3017,0.3984
4. Muestra (Modelo significativo),15,0.4124,0.4004,0.3055,0.4048


* Tanto el MAE como el RMSE permanecen inalterados en el tercer decimal al reducir las variables. Quitar variables no significativas no degrad0 la precision predictiva.   

* En los cuatro modelos, el RMSE es mayor que el MAE, lo cual demuestra la presencia de alquileres con errores extremos (propiedades atípicas o fuera de mercado que el modelo subestima o sobreestima fuertemente).

* Como la variable dependiente esta expresada en escala logaritmica, un MAE de 0.30 indica que las predicciones del modelo se desvian, en promedio, aproximadamente un 30% respecto al valor real de la publicacion.  

* Poblacion (1 vs. 2): Al pasar de 33 a 27 variables, el R2 cayo de 0.3979 a 0.3974 (una baja insignificante de 0.05), mientras que el R2 ajustado se mantiene casi identico (0.3994 vs 0.3995). Esto comprueba que las 6 variables eliminadas aportaban unicamente ruido aleatorio y nula capacidad explicativa real.
* Muestra (3 vs. 4): Se eliminaron 18 variables (más del 50% de los predictores). Aun asi, el R2 ajustado solo vario de 0.4046 a 0.4004 (menos de medio punto porcentual). El modelo reducido conserva el 99% de la potencia explicativa ajustada del modelo saturado con menos de la mitad de las variables.

Comparando la muestra con el total del dataset EDA, podemos decir que:
* En la población de 9297 propiedades, casi cualquier efecto pequeño resulta estadisticamente significativo. En la muestra de 750, los errores estandar son mayores y las variables como amenities específicas (cafetera, plancha, ascensor) pierden significancia.

Las variables que se sostuvieron como significativas tanto en la poblacion como en la muestra son el nucleo explicativo fundamental del precio de un Airbnb en Milan:
* Ubicacion: neighbourhood_cleansed
* Capacidad del alojamiento: accommodates, bathrooms, bedrooms
* Reputacion y perfil del host: host_total_listings_count, number_of_reviews, review_scores_rating.
* Conectividad: WiFi.